# Gemma: Jungle — zero-shot and few-shot

Historical experiment source: `bench_gemma/FewZeroShot/LLM_Gemma3_4B_Zero_Few_shot_with_Jungle_multi (1).ipynb`. Outputs and stale result commentary were removed for publication. Scientific logic is retained; these experiments and their reported results have not been rerun or validated here.

In [ ]:
!pip install -U \
  torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu128

!pip install -U \
  "transformers>=4.51.3" \
  "accelerate>=1.4.0" \
  "peft>=0.14.0" \
  "bitsandbytes>=0.45.3" \
  "datasets>=3.3.2" \
  scikit-learn ucimlrepo sentencepiece protobuf huggingface_hub

In [ ]:
import os
from huggingface_hub import login
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
import os
import gc
import time
import json
import math
import random
from pathlib import Path

import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

BASE_DIR = Path.cwd() / "gemma_jungle_multi"
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs" / "gemma-3-4b-it-base"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Рабочая директория: {BASE_DIR}")
print(f"Директория для модели: {OUTPUT_DIR}")

In [ ]:
# Настройка промпта для Jungle

prompt_config = {
    "task": "Predict the endgame result of Jungle Chess (Dou Shou Qi)",
    "labels": ["white_win", "draw", "black_win"],
    "entity": "Game Position",
    "question": "Based on the rank, file, and strength of the white and black pieces, what is the game result? (White wins, Black wins, or Draw)"
}

openml_id = 41027

# 1. Загрузка данных

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464, prompt_config=None):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    # Преобразование целевой переменной в бинарный формат
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])
        class_names = prompt_config['labels']
    else:
        class_names = sorted(df[target_name].unique().tolist())

    return df, feature_names, target_name, class_names

def split_dataset(df, target_name, test_size=0.2, seed=42):

    # Разделение на train/test (80/20)
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    return train_df, test_df

df, feature_names, target_name, class_names = load_dataset(openml_id, prompt_config)
train_df, test_df = split_dataset(df, target_name)

In [ ]:
df[target_name].value_counts()

In [ ]:
df.info()

In [ ]:
for name, df in [("train", train_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")

    for i, label in enumerate(prompt_config['labels']):
        count = counts.get(i, 0)
        pct = pcts.get(i, 0.0)
        print(f"  {label:10s}: {count:5d} — {pct:.1f}%")

# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names, target_name, prompt_config=None, include_target=False):
    template_parts = []

    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    if include_target and prompt_config is not None:
        target_value = prompt_config['labels'][int(row[target_name])]
        text += f": {target_name} -> {target_value}"

    return text

# Тест
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, True))
print(row_to_text_template(train_df.iloc[0], feature_names, target_name, prompt_config, False))
train_df.head(1)

In [ ]:
def parse_prediction(response, prompt_config):
    """Парсинг ответа модели в номер класса"""
    response = response.lower().strip()
    response = response.rstrip('.,!? ')

    # Проверка каждого класса
    for i, class_name in enumerate(prompt_config['labels']):
        class_lower = class_name.lower()

        # Точное совпадение
        if response == class_lower:
            return i

        # Начинается с имени класса
        if response.startswith(class_lower):
            return i

        # Содержит как отдельное слово
        if class_lower in response.split():
            return i

    # Не удалось распознать - возвращаем первый класс
    print(f"Warning: Could not parse '{response}' (expected one of {prompt_config['labels']})")
    return 0

response = "good"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "no"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
# Вычисление метрик качества
def compute_metrics(y_true, y_pred, y_prob=None):
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    pr = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)

    if y_prob is not None:
        try:
            roc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
        except:
            roc = 0.0
    else:
        roc = 0.0
    return roc, f1, acc, pr, rec

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    """Bootstrap метрики с доверительными интервалами"""
    scores = []

    for i in range(n_iter):
        # Bootstrap выборка
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            # Вычисление метрик
            acc = accuracy_score(y_true_boot, y_pred_boot)
            f1 = f1_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, average="macro", zero_division=0)

            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot, multi_class="ovr", average="macro")
            else:
                auc = 0.0

            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}±{s:.4f}" for n, m, s in zip(names, means, stds)}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model_name = "google/gemma-3-4b-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"Using device: {device}")
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    torch_dtype=compute_dtype,
    device_map={"": 0} if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to(device)

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    reserved_mem = torch.cuda.memory_reserved() / 1e9
    allocated_mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU total memory: {total_mem:.2f} GB")
    print(f"GPU reserved memory: {reserved_mem:.2f} GB")
    print(f"GPU allocated memory: {allocated_mem:.2f} GB")

In [ ]:
def create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples=None):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"Answer with only one word from: '{labels_str}'."
    )

    if few_shot_examples is None:
        # Zero-shot промпт
        user_message = (
            f"{prompt_config['entity']} information: "
            f"{row_to_text_template(row, feature_names, target_name, prompt_config)}\n"
            f"{prompt_config['question']}"
        )
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    else:
        # Few-shot промпт
        messages = [{"role": "system", "content": system_prompt}]

        for ex in few_shot_examples:
            ex_text   = row_to_text_template(ex, feature_names, target_name, prompt_config)
            ex_target = prompt_config['labels'][int(ex[target_name])]

            messages.append({
                "role": "user",
                "content": f"{prompt_config['entity']} information: {ex_text}\n{prompt_config['question']}"
            })
            messages.append({
                "role": "assistant",
                "content": ex_target
            })

        client_text = row_to_text_template(row, feature_names, target_name, prompt_config)
        messages.append({
            "role": "user",
            "content": f"{prompt_config['entity']} information: {client_text}\n{prompt_config['question']}"
        })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # добавление role assistant
    )

In [ ]:
import torch.nn.functional as F

BATCH_SIZE_ZERO_SHOT = 16
BATCH_SIZE_FEW_SHOT  = 8

def batched_rows(df, batch_size):
    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start + batch_size]

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def predict_batch_with_prob(prompts, prompt_config, model, tokenizer, device, max_new_tokens=3):
    model_inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )

    input_len = model_inputs.input_ids.shape[1]
    del model_inputs

    generated_ids = outputs.sequences[:, input_len:]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    responses = [resp.strip().lower() for resp in responses]
    del generated_ids

    first_token_logits = outputs.scores[0]
    del outputs

    labels_ids = [tokenizer.encode(label, add_special_tokens=False)[0] for label in prompt_config['labels']]
    labels_logits = torch.stack([first_token_logits[:, cid] for cid in labels_ids], dim=1)
    del first_token_logits

    probs = F.softmax(labels_logits, dim=1)
    probs_np = probs.detach().cpu().numpy()
    del labels_logits, probs

    flush_gpu()

    return responses, probs_np

# Тест на маленьком батче
test_prompts = [
    create_prompt(test_df.iloc[i], feature_names, target_name, prompt_config, tokenizer)
    for i in range(2)
]
responses, probs = predict_batch_with_prob(test_prompts, prompt_config, model, tokenizer, device)
print(f"Response: '{responses[0]}'")
print(f"Probabilities: {dict(zip(prompt_config['labels'], probs[0]))}")

# 3.1 Zero-shot

In [ ]:
print("Zero-shot классификация")
flush_gpu()

seed = 42
n_batches = math.ceil(len(test_df) / BATCH_SIZE_ZERO_SHOT)

y_true_zero = []
y_pred_zero = []
y_prob_zero = []

start_time = time.time()

for batch_df in tqdm(batched_rows(test_df, BATCH_SIZE_ZERO_SHOT), total=n_batches):
    prompts = [
        create_prompt(row, feature_names, target_name, prompt_config, tokenizer)
        for _, row in batch_df.iterrows()
    ]
    responses, probs = predict_batch_with_prob(prompts, prompt_config, model, tokenizer, device)

    for (_, row), response, prob in zip(batch_df.iterrows(), responses, probs):
        prediction = parse_prediction(response, prompt_config)
        y_true_zero.append(row[target_name])
        y_pred_zero.append(prediction)
        y_prob_zero.append(prob)

    del prompts, responses, probs

zero_shot_time = time.time() - start_time

print(f"zero_shot_time: {zero_shot_time:.1f}s")
print(f"per sample: {zero_shot_time / len(y_true_zero):.4f}s")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Вычисляем метрики
roc_zero, f1_zero, acc_zero, pr_zero, rec_zero = compute_metrics(np.array(y_true_zero), np.array(y_pred_zero), np.array(y_prob_zero))

print("Результаты zero-shot:")
print(f"ROC AUC: {roc_zero}")
print(f"F1 Score: {f1_zero}")
print(f"Accuracy: {acc_zero}")
print(f"Precision: {pr_zero}")
print(f"Recall: {rec_zero}")

In [ ]:
zero_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_zero),
    np.array(y_pred_zero),
    np.array(y_prob_zero),
    n_iter=1000
)

print("\nРезультаты zero-shot (bootstrap метрики с доверительными интервалами):")
for key, value in zero_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")

# 3.2 Few-shot

In [ ]:
n_examples = 64
seed = 42

num_classes = len(prompt_config['labels'])
n_per_class = n_examples // num_classes

sampled_dfs = []
for cls_idx in range(num_classes):
    cls_examples = train_df[train_df[target_name] == cls_idx].sample(n=n_per_class, random_state=seed)
    sampled_dfs.append(cls_examples)

few_shot_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed).reset_index(drop=True)

print(f"\nВыбрано {len(few_shot_df)} примеров для контекста:")
for cls_idx, label in enumerate(prompt_config['labels']):
    count = (few_shot_df[target_name] == cls_idx).sum()
    print(f"  {label}: {count}")

few_shot_examples = [row for _, row in few_shot_df.iterrows()]

flush_gpu()
print(f"GPU перед few-shot: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

y_true_few = []
y_pred_few = []
y_prob_few = []

n_batches = math.ceil(len(test_df) / BATCH_SIZE_FEW_SHOT)
start_time = time.time()

for batch_df in tqdm(batched_rows(test_df, BATCH_SIZE_FEW_SHOT), total=n_batches):
    prompts = [
        create_prompt(row, feature_names, target_name, prompt_config, tokenizer, few_shot_examples)
        for _, row in batch_df.iterrows()
    ]
    responses, probs = predict_batch_with_prob(prompts, prompt_config, model, tokenizer, device)

    for (_, row), response, prob in zip(batch_df.iterrows(), responses, probs):
        prediction = parse_prediction(response, prompt_config)
        y_true_few.append(row[target_name])
        y_pred_few.append(prediction)
        y_prob_few.append(prob)

    del prompts, responses, probs

few_shot_time = time.time() - start_time

print(f"few_shot_time: {few_shot_time:.1f}s")
print(f"per sample: {few_shot_time / len(y_true_few):.4f}s")
print(f"GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
roc_few, f1_few, acc_few, pr_few, rec_few = compute_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few)
)
print("Результаты few-shot:")
print(f"ROC AUC: {roc_few}")
print(f"F1 Score: {f1_few}")
print(f"Accuracy: {acc_few}")
print(f"Precision: {pr_few}")
print(f"Recall: {rec_few}")


In [ ]:
few_shot_metrics_bootstrap = bootstrap_metrics(
    np.array(y_true_few),
    np.array(y_pred_few),
    np.array(y_prob_few),
    n_iter=1000
)

print("\nРезультаты few-shot (bootstrap метрики с доверительными интервалами):")
for key, value in few_shot_metrics_bootstrap.items():
    print(f"  {key}: {value}")